<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


# Docker Behind the Duckietown Shell

By now, we have built images, started containers, shared files, and selected Docker hosts. Doing this manually helps explain how Docker works, but a robotics project may need many settings every time it runs.

The **Duckietown Shell**, invoked with `dts`, automates recurring Docker operations using project configuration and Duckietown conventions. You still work with images, containers, and Docker hosts; the shell abstracts much of the detail and handles most of the setup.

This notebook connects three Duckietown workflows to the Docker concepts underneath them.

## Building and running projects with `dts devel`

Deploying a Duckietown project on a robot usually involves building and running Docker containers. With plain Docker, you have to supply a build context, choose an image tag, and configure the container's startup command and mounts. For a Duckietown project (a [`DTProject`](https://docs.duckietown.com/ente/duckietown-manual/70-developer-manual/dtproject/introduction-to-duckietown-projects.html)), `dts devel` assembles these settings from the project configuration and command options.

From the root of a configured Duckietown project, the basic workflow is:

```bash
dts devel build
dts devel run
```

The first command builds the project image. The second starts a container using that image and the project's runtime settings. The Docker host still matters. So, for example, `dts devel` would be used with: 

```bash
dts devel build -H DUCKIEDRONE_NAME
dts devel run -H DUCKIEDRONE_NAME
```

Here, `-H` selects the machine used for the operation. Building on the Duckiedrone uses its resources and places the image in its daemon's inventory; running there starts the container there.

Building and running are separate operations. A successful build does not mean that the application is running.

### Explore the wrapper

Inspect the options available in your installed version:

```bash
dts devel build --help
dts devel run --help
```

Find `--machine` in both commands and `--destination` in the build command.

What distinction do they express?

- The **build machine** performs the build.
- The **destination** receives the resulting image.
- The **run machine** starts the container.

These can be the same machine, but they need not be. When building for another machine, the image must support its processor architecture.

The `dts` host options are separate from Docker context names. A context created for `docker --context` is not automatically a target recognized by `dts -H`.

## Preparing LX environments with `dts code`

Learning experiences use project-specific environments too. A **recipe** supplies the build instructions and configuration needed for an LX, while a **launcher** starts a particular project process inside its container.

For an LX that provides a robot workbench, its instructions may use:

```bash
dts code build -R DUCKIEDRONE_NAME
dts code workbench -R DUCKIEDRONE_NAME
```

The build command prepares the recipe and delegates the image build to `dts devel build`. The workbench command uses `dts devel run` to start the project with the required runtime configuration.

The usual `-R` form names the robot and targets it for these operations. Follow the particular LX's instructions when it supplies additional options, such as `--local`.

Once the workbench is running, a second terminal can open a shell inside it:

```bash
dts code workbench -R DUCKIEDRONE_NAME --shell
```

This connects to the existing workbench container. It does not build another image or create an independent copy of the environment.

Relate this to the earlier `docker exec` exercise: an additional shell lets you inspect files and run tools alongside the container's main process.

You can verify the relationship in the Duckietown Shell's [build implementation](https://github.com/duckietown/duckietown-shell-commands/blob/312fa45782ec700c3e27c62dc193c8bcc22439f8/code/build/command.py) and [workbench implementation](https://github.com/duckietown/duckietown-shell-commands/blob/312fa45782ec700c3e27c62dc193c8bcc22439f8/code/workbench/command.py). Search for `shell.include.devel`: these calls show the higher-level commands delegating work.

## Entering a virtual Duckiedrone

A virtual Duckiedrone provides another example of Docker underneath a familiar command.

For an existing, running virtual Duckiedrone:

```bash
dts duckiebot virtual connect DUCKIEDRONE_NAME
```

The [connection implementation](https://github.com/duckietown/duckietown-shell-commands/blob/312fa45782ec700c3e27c62dc193c8bcc22439f8/duckiebot/virtual/connect/command.py) opens a shell using the equivalent of:

```bash
docker exec -it dts-virtual-DUCKIEDRONE_NAME /bin/bash
```

The shell enters a container on the base station. This differs from opening an SSH shell on a physical Duckiedrone.

If you already have a virtual Duckiedrone available, connect and inspect its services:

```bash
docker container ls
exit
```

Inside that environment, Docker commands query the virtual Duckiedrone's internal daemon. Its inventory is separate from the base station daemon's inventory, which contains the outer virtual-robot container.

This is a deliberately configured container that runs Docker internally. Ordinary practice containers do not automatically contain their own daemon.

## Why learn Docker if `dts` handles it?

The wrapper reduces repeated configuration, but Docker knowledge helps you interpret failures, plan improvements, and design experiments.

If a project builds but does not run, ask whether its image reached the intended daemon. If a workbench cannot find a file, examine how the project supplies files to the container. If a shell shows unexpected containers, check which daemon it is querying.

You do not need to reconstruct every `dts` command manually. You need to recognize the image, container, mount, and host involved so you can investigate the right part of the workflow.

## Further reading

Use the installed command help for the options available in your environment:

```bash
dts code build --help
dts code workbench --help
dts duckiebot virtual --help
```

Follow each LX's own setup and execution instructions for its recipe, launcher, and robot requirements.

## Checkpoint

Before revealing the answers, consider:

1. What Docker object does a build produce, and what does running it create?
2. Why might an image built successfully on the base station be unavailable on the Duckiedrone?
3. Does opening a workbench shell create a second workbench container?
4. What does `dts duckiebot virtual connect` do underneath?

Run the self-check in the next cell.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()


## Summary and conclusions

The Duckietown Shell combines Docker operations with project configuration to build images, run applications, and open development shells. Recognizing the image, container, and daemon underneath each command helps explain where the work happens. Finally, [inspect the Duckietown Workspace](./12-development-containers-and-duckietown-workspaces.ipynb) as a development environment built with Docker.
